#Differential Expression with DESeq2
   RNA-Sequence Analysis Workflow

1. Install packages and load libraries
2. Read gene counts and meta data into a data frame
3. Quality assess and clean raw sequencing data
4. Align reads to a reference
5. Count the number of reads assigned to each contig/gene
6. Extract counts and store in a matrix
7. Create column metadata table
8. Analyze count data using DESEQ2

`Sk. Tanzir Mehedi`

`Lecturer, Department of IT, UITS`


Install package `Biobase`

In [1]:
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

BiocManager::install("Biobase")

Bioconductor version 3.12 (BiocManager 1.30.10), R 4.0.4 (2021-02-15)

Installing package(s) 'Biobase'



package 'Biobase' successfully unpacked and MD5 sums checked


Warning message:
"cannot remove prior installation of package 'Biobase'"
Warning message in file.copy(savedcopy, lib, recursive = TRUE):
"problem copying C:\Users\shuvo\OneDrive\Documents\R\win-library\4.0\00LOCK\Biobase\libs\x64\Biobase.dll to C:\Users\shuvo\OneDrive\Documents\R\win-library\4.0\Biobase\libs\x64\Biobase.dll: Permission denied"
Warning message:
"restored 'Biobase'"



The downloaded binary packages are in
	C:\Users\shuvo\AppData\Local\Temp\RtmpWopeRb\downloaded_packages


Installation path not writeable, unable to update packages: boot, cluster,
  MASS, mgcv, survival

Old packages: 'BiocManager', 'broom', 'caTools', 'cli', 'cpp11',
  'DelayedArray', 'RCurl', 'RSQLite', 'tinytex', 'utf8', 'vctrs'



Install package `DESeq2`

In [2]:
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

BiocManager::install("DESeq2")

Bioconductor version 3.12 (BiocManager 1.30.10), R 4.0.4 (2021-02-15)

Installing package(s) 'DESeq2'



package 'DESeq2' successfully unpacked and MD5 sums checked


Warning message:
"cannot remove prior installation of package 'DESeq2'"
Warning message in file.copy(savedcopy, lib, recursive = TRUE):
"problem copying C:\Users\shuvo\OneDrive\Documents\R\win-library\4.0\00LOCK\DESeq2\libs\x64\DESeq2.dll to C:\Users\shuvo\OneDrive\Documents\R\win-library\4.0\DESeq2\libs\x64\DESeq2.dll: Permission denied"
Warning message:
"restored 'DESeq2'"



The downloaded binary packages are in
	C:\Users\shuvo\AppData\Local\Temp\RtmpWopeRb\downloaded_packages


Installation path not writeable, unable to update packages: boot, cluster,
  MASS, mgcv, survival

Old packages: 'BiocManager', 'broom', 'caTools', 'cli', 'cpp11',
  'DelayedArray', 'RCurl', 'RSQLite', 'tinytex', 'utf8', 'vctrs'



Load libraries

In [3]:
library(DESeq2)
library(ggplot2)
library(dplyr)
library(readr)
library(RColorBrewer)

Loading required package: S4Vectors

Loading required package: stats4

Loading required package: BiocGenerics

Loading required package: parallel


Attaching package: 'BiocGenerics'


The following objects are masked from 'package:parallel':

    clusterApply, clusterApplyLB, clusterCall, clusterEvalQ,
    clusterExport, clusterMap, parApply, parCapply, parLapply,
    parLapplyLB, parRapply, parSapply, parSapplyLB


The following objects are masked from 'package:stats':

    IQR, mad, sd, var, xtabs


The following objects are masked from 'package:base':

    anyDuplicated, append, as.data.frame, basename, cbind, colnames,
    dirname, do.call, duplicated, eval, evalq, Filter, Find, get, grep,
    grepl, intersect, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, setdiff, sort, table, tapply,
    union, unique, unsplit, which.max, which.min



Attaching package: 'S4Vectors'


The follow

Read gene counts data

In [4]:
countData <- read.csv('GSE52463_filtered_countdata.csv', header = TRUE, sep = ",")
head(countData)

,X,GSM1267256,GSM1267257,GSM1267258,GSM1267259,GSM1267260,GSM1267261,GSM1267262,GSM1267263,GSM1267264,GSM1267265,GSM1267266,GSM1267267,GSM1267268,GSM1267269,GSM1267270
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,TSPAN6,14.24430,15.02526,27.76140,11.07250,19.13955,15.61760,25.13495,15.16884,20.64407,23.43800,42.76766,46.99986,18.69651,13.88022,10.36363
2,TNMD,0.00000,0.00000,0.12334,0.10909,0.22466,0.71871,0.56166,0.37937,0.41102,0.37339,0.13659,0.30473,0.19631,0.39769,0.06039
3,DPM1,14.29805,15.76512,20.77851,20.54884,10.50701,14.21862,13.14437,15.09053,15.01098,13.44716,15.73708,16.17674,11.91389,9.66573,8.75745
4,SCYL3,16.80300,23.56145,22.17744,20.25619,21.14408,18.89588,20.00938,13.62947,19.36906,16.80407,20.86146,22.70301,13.35902,23.94934,21.00449
5,C1orf112,4.88083,6.78888,8.97993,4.61197,7.15457,7.82400,9.23296,6.15785,5.41373,6.18505,5.70548,7.02731,5.25408,6.06814,6.09974
6,FGR,82.32020,78.58903,35.10407,97.44321,66.35449,23.92128,28.09829,35.60068,40.09447,48.60954,49.56289,72.05253,81.48779,36.77666,45.63506


Check row and colums of gene counts data

In [5]:
ncol(countData)
nrow(countData)

[1] 16

[1] 28089

Transform gene counts data into data frame

In [6]:
countDataFrame <- data.frame(countData)
head(countDataFrame)

,X,GSM1267256,GSM1267257,GSM1267258,GSM1267259,GSM1267260,GSM1267261,GSM1267262,GSM1267263,GSM1267264,GSM1267265,GSM1267266,GSM1267267,GSM1267268,GSM1267269,GSM1267270
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,TSPAN6,14.24430,15.02526,27.76140,11.07250,19.13955,15.61760,25.13495,15.16884,20.64407,23.43800,42.76766,46.99986,18.69651,13.88022,10.36363
2,TNMD,0.00000,0.00000,0.12334,0.10909,0.22466,0.71871,0.56166,0.37937,0.41102,0.37339,0.13659,0.30473,0.19631,0.39769,0.06039
3,DPM1,14.29805,15.76512,20.77851,20.54884,10.50701,14.21862,13.14437,15.09053,15.01098,13.44716,15.73708,16.17674,11.91389,9.66573,8.75745
4,SCYL3,16.80300,23.56145,22.17744,20.25619,21.14408,18.89588,20.00938,13.62947,19.36906,16.80407,20.86146,22.70301,13.35902,23.94934,21.00449
5,C1orf112,4.88083,6.78888,8.97993,4.61197,7.15457,7.82400,9.23296,6.15785,5.41373,6.18505,5.70548,7.02731,5.25408,6.06814,6.09974
6,FGR,82.32020,78.58903,35.10407,97.44321,66.35449,23.92128,28.09829,35.60068,40.09447,48.60954,49.56289,72.05253,81.48779,36.77666,45.63506


Transform gene counts data into integer for processing

In [8]:
countDataFrameRound <-countDataFrame %>% mutate(across(where(is.numeric), round,3))
head(countDataFrameRound)

,X,GSM1267256,GSM1267257,GSM1267258,GSM1267259,GSM1267260,GSM1267261,GSM1267262,GSM1267263,GSM1267264,GSM1267265,GSM1267266,GSM1267267,GSM1267268,GSM1267269,GSM1267270
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,TSPAN6,14.244,15.025,27.761,11.072,19.140,15.618,25.135,15.169,20.644,23.438,42.768,47.000,18.697,13.880,10.364
2,TNMD,0.000,0.000,0.123,0.109,0.225,0.719,0.562,0.379,0.411,0.373,0.137,0.305,0.196,0.398,0.060
3,DPM1,14.298,15.765,20.779,20.549,10.507,14.219,13.144,15.091,15.011,13.447,15.737,16.177,11.914,9.666,8.757
4,SCYL3,16.803,23.561,22.177,20.256,21.144,18.896,20.009,13.629,19.369,16.804,20.861,22.703,13.359,23.949,21.004
5,C1orf112,4.881,6.789,8.980,4.612,7.155,7.824,9.233,6.158,5.414,6.185,5.705,7.027,5.254,6.068,6.100
6,FGR,82.320,78.589,35.104,97.443,66.354,23.921,28.098,35.601,40.094,48.610,49.563,72.053,81.488,36.777,45.635


Read meta data

In [9]:
metaData <- read.csv('GSE52463_filtered_metadata.csv', header = TRUE, sep = ",")
head(metaData)

,X,treatment
,<chr>,<chr>
1,GSM1267256,Norm1
2,GSM1267257,Norm2
3,GSM1267258,Norm3
4,GSM1267259,Norm4
5,GSM1267260,Norm5
6,GSM1267261,Norm6


Check row and colums of meta data

In [36]:
colnames(metaData)
ncol(metaData)
nrow(metaData)
ncol(countData)

[1] "X"         "treatment"

[1] 2

[1] 15

[1] 16

Transform meta data into data frame

In [11]:
metaDataFrame <- data.frame(metaData, row.names=1)
head(metaDataFrame)

,treatment
,<chr>
GSM1267256,Norm1
GSM1267257,Norm2
GSM1267258,Norm3
GSM1267259,Norm4
GSM1267260,Norm5
GSM1267261,Norm6


Check gene counts and meta data dimension

In [12]:
rownames(metaDataFrame)
colnames(countDataFrameRound)

[1] "GSM1267256" "GSM1267257" "GSM1267258" "GSM1267259" "GSM1267260"
 [6] "GSM1267261" "GSM1267262" "GSM1267263" "GSM1267264" "GSM1267265"
[11] "GSM1267266" "GSM1267267" "GSM1267268" "GSM1267269" "GSM1267270"

[1] "X"          "GSM1267256" "GSM1267257" "GSM1267258" "GSM1267259"
 [6] "GSM1267260" "GSM1267261" "GSM1267262" "GSM1267263" "GSM1267264"
[11] "GSM1267265" "GSM1267266" "GSM1267267" "GSM1267268" "GSM1267269"
[16] "GSM1267270"

Run DESeq2 model

In [14]:
dds <- DESeqDataSetFromMatrix(countData=countDataFrameRound, colData=metaDataFrame, design=~treatment, tidy=FALSE)
dds

ERROR: Error in DESeqDataSetFromMatrix(countData = countDataFrameRound, colData = metaDataFrame, : ncol(countData) == nrow(colData) is not TRUE


Run DESEQ function for fitting and testin the model

In [43]:
dds <- DESeq(dds)

estimating size factors

estimating dispersions



ERROR: Error in checkForExperimentalReplicates(object, modelMatrix): 

  The design matrix has the same number of samples and coefficients to fit,
  so estimation of dispersion is not possible. Treating samples
  as replicates was deprecated in v1.20 and no longer supported since v1.22.




Take a look at the results table

In [15]:
res <- results(dds)
head(results(dds, tidy=TRUE))

ERROR: Error in is(object, "DESeqDataSet"): object 'dds' not found


Summary of differential gene expression

In [16]:
summary(res)

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'object' in selecting a method for function 'summary': object 'res' not found


Write DESeq2 data to file

In [17]:
write.csv(res, file = "GSE52463_results.csv")

ERROR: Error in is.data.frame(x): object 'res' not found


#Second Part: Results Analysis

Read results data

In [18]:
results <- read.csv("GSE52463_results.csv",header=T, sep=',')
head(results)

Warning message in file(file, "rt"):
"cannot open file 'GSE52463_results.csv': No such file or directory"


ERROR: Error in file(file, "rt"): cannot open the connection


Check null value

In [19]:
check_null <- is.na(results)
head(check_null)

Warning message in is.na(results):
"is.na() applied to non-(list or vector) of type 'closure'"


[1] FALSE

Omit the null value

In [20]:
results_omit_na <- na.omit(results)
head(results_omit_na)

                                                                                      
1 function (object, contrast, name, lfcThreshold = 0, altHypothesis = c("greaterAbs", 
2     "lessAbs", "greater", "less"), listValues = c(1, -1), cooksCutoff,              
3     independentFiltering = TRUE, alpha = 0.1, filter, theta,                        
4     pAdjustMethod = "BH", filterFun, format = c("DataFrame",                        
5         "GRanges", "GRangesList"), test, addMLE = FALSE, tidy = FALSE,              
6     parallel = FALSE, BPPARAM = bpparam(), minmu = 0.5)                             

Count the up regulated gene

In [21]:
results_omit_na_filter_up <- filter(results_omit_na, log2FoldChange>1 & padj<0.05)
head(results_omit_na_filter_up)
nrow(results_omit_na_filter_up)

ERROR: Error in UseMethod("filter"): no applicable method for 'filter' applied to an object of class "function"


Count the down regulated gene

In [22]:
results_omit_na_filter_down <- filter(results_omit_na, log2FoldChange<-1 & padj<0.05)
head(results_omit_na_filter_down)
nrow(results_omit_na_filter_down)

ERROR: Error in UseMethod("filter"): no applicable method for 'filter' applied to an object of class "function"


ABS logFC value and setup cuttoff criteria for p value

In [23]:
results_omit_na_filter <- filter(results_omit_na, abs(log2FoldChange)>1 & pvalue<0.05)
head(results_omit_na_filter)
nrow(results_omit_na_filter)

ERROR: Error in UseMethod("filter"): no applicable method for 'filter' applied to an object of class "function"


ABS logFC value and setup cuttoff criteria for p adj value

In [24]:
results_omit_na_filter <- filter(results_omit_na, abs(log2FoldChange)>1 & padj<0.05)
head(results_omit_na_filter)
nrow(results_omit_na_filter)

ERROR: Error in UseMethod("filter"): no applicable method for 'filter' applied to an object of class "function"


Write DESeq2 final results to file

In [25]:
write.csv(results_omit_na_filter, file="final_result_GSE52463.csv")

ERROR: Error in is.data.frame(x): object 'results_omit_na_filter' not found


#Third Part: Visualization (graphics)

Read final results

In [26]:
resultsShow<- read.csv("final_result_GSE52463.csv",header=T, sep=',')
head(resultsShow)

Warning message in file(file, "rt"):
"cannot open file 'final_result_GSE52463.csv': No such file or directory"


ERROR: Error in file(file, "rt"): cannot open the connection


#Sort summary list by p-value

In [27]:
res <- res[order(res$padj),]
head(res)

ERROR: Error in eval(expr, envir, enclos): object 'res' not found


#Plot Counts
We can use plotCounts function to compare the normalized counts between treated and control groups for our top 6 genes

In [28]:
par(mfrow=c(2,3))

plotCounts(dds, gene="TMEM204", intgroup="treatment")
plotCounts(dds, gene="SLC44A4", intgroup="treatment")
plotCounts(dds, gene="DUOX1", intgroup="treatment")
plotCounts(dds, gene="JAM2", intgroup="treatment")
plotCounts(dds, gene="CCL3", intgroup="treatment")
plotCounts(dds, gene="ADIRF", intgroup="treatment")

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function 'nrow': object 'dds' not found


Next steps in exploring these data...BLAST to database to find associated gene function

#Volcano Plot

In [29]:
#reset par
par(mfrow=c(1,1))
# Make a basic volcano plot
with(res, plot(log2FoldChange, -log10(pvalue), pch=20, main="Volcano plot", xlim=c(-3,3)))

# Add colored points: blue if padj<0.01, red if log2FC>1 and padj<0.05)
with(subset(res, padj<.05 ), points(log2FoldChange, -log10(pvalue), pch=20, col="blue"))
with(subset(res, padj<.05 & abs(log2FoldChange)>2), points(log2FoldChange, -log10(pvalue), pch=20, col="red"))

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'data' in selecting a method for function 'with': object 'res' not found


#PCA
First we need to transform the raw count data vst function will perform variance stabilizing transformation

Using the DESEQ2 plotPCA fxn we can

In [30]:
vsdata <- vst(dds, blind=FALSE)
plotPCA(vsdata, intgroup="treatment")

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function 'nrow': object 'dds' not found


Extract results for the top 250 up-regulated and top 250 down-regulated genes, sorted by p-value:

Print results for top and bottom 5 genes

In [31]:
n = 250 
resOrdered <- results_omit_na[order(results_omit_na$padj),]
topResults <- rbind( resOrdered[ resOrdered[,'log2FoldChange'] > 0, ][1:n,], resOrdered[ resOrdered[,'log2FoldChange'] < 0, ][n:1,] )
topResults[c(1:5,(2*n-4):(2*n)), c('X','baseMean','log2FoldChange','padj')]

ERROR: Error in results_omit_na$padj: object of type 'closure' is not subsettable


Plot counts for a single gene. Below is the plot for the gene with the lowest p-value:

In [32]:
plotCounts(dds, gene=which.min(results_omit_na$padj), intgroup='treatment', pch = 19)

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function 'which.min': object of type 'closure' is not subsettable
